<a href="https://colab.research.google.com/github/Akspaks007/DL-Lab/blob/main/Exp_4/Exp_4_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data Processing

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader

DATA_PATH = '/content/drive/MyDrive/ML lab file/poems-100.csv'
df = pd.read_csv(DATA_PATH)
full_text = " ".join(df['text'].astype(str).tolist())

def preprocess_text(text):
    words = text.lower().split()
    vocab = sorted(list(set(words)))
    word_to_int = {word: i for i, word in enumerate(vocab)}
    int_to_word = {i: word for i, word in enumerate(vocab)}

    encoded = [word_to_int[w] for w in words]
    return encoded, vocab, word_to_int, int_to_word

encoded, vocab, word_to_int, int_to_word = preprocess_text(full_text)
vocab_size = len(vocab)

print(f"Vocabulary Size: {vocab_size}")

def get_one_hot(batch, vocab_size):
    return torch.nn.functional.one_hot(batch, num_classes=vocab_size).float()

Vocabulary Size: 6989


Dataset Class

In [3]:
class TextDataset(Dataset):
    def __init__(self, encoded, seq_length):
        self.encoded = encoded
        self.seq_length = seq_length

    def __len__(self):
        return len(self.encoded) - self.seq_length

    def __getitem__(self, idx):
        x = torch.tensor(self.encoded[idx : idx + self.seq_length])
        y = torch.tensor(self.encoded[idx + 1 : idx + self.seq_length + 1])
        return x, y

seq_length = 3
dataset = TextDataset(encoded, seq_length)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

RNN + LSTM Architecture

In [4]:

class RNNModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim, n_layers=1):
        super(RNNModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.rnn = nn.RNN(vocab_size, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        out, hidden = self.rnn(x, hidden)
        out = out.reshape(-1, self.hidden_dim)
        out = self.fc(out)
        return out, hidden

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim, n_layers=1):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.lstm = nn.LSTM(vocab_size, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        out, hidden = self.lstm(x, hidden)
        out = out.reshape(-1, self.hidden_dim)
        out = self.fc(out)
        return out, hidden

Training

In [5]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMModel(vocab_size, hidden_dim=128).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

model.train()
for epoch in range(30):
    for x, y in loader:
        x_one_hot = get_one_hot(x, vocab_size).to(device)
        y = y.view(-1).to(device)

        optimizer.zero_grad()
        output, _ = model(x_one_hot, None)

        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")

Epoch 5 | Loss: 1.7912
Epoch 10 | Loss: 2.6664
Epoch 15 | Loss: 0.1776
Epoch 20 | Loss: 1.2252
Epoch 25 | Loss: 0.6553
Epoch 30 | Loss: 1.8988
Epoch 35 | Loss: 4.5947
Epoch 40 | Loss: 0.0234
Epoch 45 | Loss: 0.3975
Epoch 50 | Loss: 1.4173


Text Generation

In [23]:

def generate(model, start_str, length, vocab_size, word_to_int, int_to_word):
    model.eval()
    words = start_str.lower().split()
    hidden = None

    for _ in range(length):
        x = torch.tensor([[word_to_int[w] for w in words[-seq_length:]]])
        x_one_hot = get_one_hot(x, vocab_size).to(device)

        output, hidden = model(x_one_hot, hidden)

        last_word_logits = output[-1]
        p = torch.nn.functional.softmax(last_word_logits, dim=0).detach().cpu().numpy()
        word_idx = np.random.choice(len(last_word_logits), p=p)

        words.append(int_to_word[word_idx])

    return ' '.join(words)

print(generate(model, "smiling at him", 40, vocab_size, word_to_int, int_to_word))

smiling at him they straying mighty set beat o wake place. i koboo crowd earth i family murderous routine it like tree. common pearl, free cut apart so, thou, mouth well outposts such love, absorbing years gross, gone crack freshly has. drops prairie-life,


TRAINABLE WORD EMBEDDINGS APPROACH

Data Processing

In [8]:

class EmbeddingDataset(Dataset):
    def __init__(self, encoded, seq_length):
        self.encoded = encoded
        self.seq_length = seq_length

    def __len__(self):
        return len(self.encoded) - self.seq_length

    def __getitem__(self, idx):
        x = torch.tensor(self.encoded[idx : idx + self.seq_length])
        y = torch.tensor(self.encoded[idx + 1 : idx + self.seq_length + 1])
        return x, y

dataset_emb = EmbeddingDataset(encoded, seq_length=3)
loader_emb = DataLoader(dataset_emb, batch_size=64, shuffle=True)

Architecture

In [9]:

class EmbeddingLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers=1):
        super(EmbeddingLSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        out, hidden = self.lstm(embedded, hidden)
        out = out.reshape(-1, out.size(2))
        out = self.fc(out)
        return out, hidden

Training

In [10]:

embed_dim = 64
model_emb = EmbeddingLSTMModel(vocab_size, embed_dim, hidden_dim=128).to(device)
optimizer = optim.Adam(model_emb.parameters(), lr=0.002)

model_emb.train()
for epoch in range(50):
    for x, y in loader_emb:
        x, y = x.to(device), y.view(-1).to(device)

        optimizer.zero_grad()
        output, _ = model_emb(x, None)

        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1} | Embedding Loss: {loss.item():.4f}")

Epoch 5 | Embedding Loss: 2.7466
Epoch 10 | Embedding Loss: 1.9677
Epoch 15 | Embedding Loss: 1.3563
Epoch 20 | Embedding Loss: 1.5540
Epoch 25 | Embedding Loss: 1.1417
Epoch 30 | Embedding Loss: 1.5166
Epoch 35 | Embedding Loss: 1.3754
Epoch 40 | Embedding Loss: 1.1216
Epoch 45 | Embedding Loss: 1.3637
Epoch 50 | Embedding Loss: 1.3209


Generation

In [20]:

def generate_emb(model, start_str, length):
    model.eval()
    words = start_str.lower().split()

    for _ in range(length):
        x = torch.tensor([[word_to_int[w] for w in words[-seq_length:]]]).to(device)
        output, _ = model(x, None)

        prob = torch.nn.functional.softmax(output[-1], dim=0).detach().cpu().numpy()
        word_idx = np.random.choice(len(prob), p=prob)

        words.append(int_to_word[word_idx])

    return ' '.join(words)

print(generate_emb(model_emb, "smiling at him", 40))

smiling at him who sings to one clear harp in divers tones, that men may rise on stepping-stones of their hides, where the cheese-cloth hangs in the kitchen, where andirons straddle the hearth-slab, where cobwebs fall in festoons from the rafters; where trip-hammers
